In [4]:
import cv2
import mediapipe as mp
import numpy as np
import json
import time

mp_hands = mp.solutions.hands

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba)*np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
    return np.degrees(np.arccos(cosine_angle))

def get_finger_angles(lm):
    return {
        'Thumb': calculate_angle([lm[2].x, lm[2].y], [lm[3].x, lm[3].y], [lm[4].x, lm[4].y]),
        'Index': calculate_angle([lm[5].x, lm[5].y], [lm[6].x, lm[6].y], [lm[8].x, lm[8].y]),
        'Middle': calculate_angle([lm[9].x, lm[9].y], [lm[10].x, lm[10].y], [lm[12].x, lm[12].y]),
        'Ring': calculate_angle([lm[13].x, lm[13].y], [lm[14].x, lm[14].y], [lm[16].x, lm[16].y]),
        'Pinky': calculate_angle([lm[17].x, lm[17].y], [lm[18].x, lm[18].y], [lm[20].x, lm[20].y])
    }

def countdown(frame, seconds=3):
    """Show countdown overlay before recording starts."""
    start = time.time()
    while time.time() - start < seconds:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)
        n = int(seconds - (time.time() - start))
        cv2.putText(frame, f"Starting in {n}...", (200, 240),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
        cv2.imshow("Calibration", frame)
        cv2.waitKey(1)

def calibrate_fingers():
    calibration = {f: {"min": 999, "max": 0} for f in ['Thumb','Index','Middle','Ring','Pinky']}
    global cap
    cap = cv2.VideoCapture(0)

    print("\n🖐 PHASE 1: Open hand calibration.")
    print("Keep your hand open and relaxed.")
    print("Press SPACEBAR to start countdown when ready.")

    phase = "idle_open"
    recording = False

    with mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.6) as hands:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.flip(frame, 1)
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(frame_rgb)

            if results.multi_hand_landmarks and recording:
                lm = results.multi_hand_landmarks[0].landmark
                angles = get_finger_angles(lm)
                if phase == "record_open":
                    for f, a in angles.items():
                        calibration[f]["min"] = min(calibration[f]["min"], a)
                elif phase == "record_closed":
                    for f, a in angles.items():
                        calibration[f]["max"] = max(calibration[f]["max"], a)

            text = "Press SPACE to start" if not recording else "Recording... press SPACE to stop"
            color = (0,255,0) if "open" in phase else (0,255,255)
            cv2.putText(frame, f"Phase: {'OPEN' if 'open' in phase else 'CLOSED'}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            cv2.putText(frame, text, (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
            cv2.imshow("Calibration", frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord(' '):
                if phase == "idle_open":
                    print("\nStarting open-hand capture in 3 seconds...")
                    countdown(frame, 3)
                    print("▶ Recording open-hand data...")
                    phase = "record_open"
                    recording = True
                elif phase == "record_open":
                    print("✅ Open-hand data recorded.")
                    print("\n✊ PHASE 2: Close your fist.")
                    print("Press SPACEBAR to start countdown when ready.")
                    phase = "idle_closed"
                    recording = False
                elif phase == "idle_closed":
                    print("\nStarting closed-hand capture in 3 seconds...")
                    countdown(frame, 3)
                    print("▶ Recording closed-hand data...")
                    phase = "record_closed"
                    recording = True
                elif phase == "record_closed":
                    print("✅ Closed-hand data recorded. Calibration complete!")
                    break
            elif key == ord('q'):
                print("❌ Calibration aborted.")
                break

    cap.release()
    cv2.destroyAllWindows()

    with open("finger_calibration.json", "w") as f:
        json.dump(calibration, f)
    print("\n✅ Calibration saved to finger_calibration.json")
    print(json.dumps(calibration, indent=2))

calibrate_fingers()



🖐 PHASE 1: Open hand calibration.
Keep your hand open and relaxed.
Press SPACEBAR to start countdown when ready.

Starting open-hand capture in 3 seconds...
▶ Recording open-hand data...
✅ Open-hand data recorded.

✊ PHASE 2: Close your fist.
Press SPACEBAR to start countdown when ready.

Starting closed-hand capture in 3 seconds...
▶ Recording closed-hand data...
✅ Closed-hand data recorded. Calibration complete!

✅ Calibration saved to finger_calibration.json
{
  "Thumb": {
    "min": 161.8330853431546,
    "max": 129.7183689563718
  },
  "Index": {
    "min": 174.4469338885387,
    "max": 19.18640552012415
  },
  "Middle": {
    "min": 178.64088790232827,
    "max": 14.286634068911352
  },
  "Ring": {
    "min": 176.1431441378083,
    "max": 16.299037998076702
  },
  "Pinky": {
    "min": 173.58462803234576,
    "max": 26.071566876035625
  }
}
